# EDA -- gold

Columns of each dim/fact in `data/gold/` and how they relate (star schema). No full dumps -- maximum 5 sample rows per table. `data/gold/ml/` not included here (predictions from `python -m src.ml.main`, not part of the star schema).

`dim_medida_trafico` and `fact_trafico_diario` were added this session (see `_catalog.yml`) to unpivot traffic metrics and aggregate straight from silver traffic (89M+ rows at 15-min grain) to a Power-BI-importable daily grain, mirroring how `dim_magnitud`/`fact_calidad_aire` already work for air quality. The old `fact_trafico` (15-min, unaggregated) was removed from gold entirely -- nothing used it once `fact_trafico_diario` existed (not Power BI, not ML).

In [3]:
import os
from pathlib import Path

import duckdb
import pandas as pd


def _repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


os.chdir(_repo_root(Path.cwd()))

DIM_DISTRITO_PATH = "data/gold/dim_distrito.parquet"
DIM_ESTACION_AIRE_PATH = "data/gold/dim_estacion_aire.parquet"
DIM_MAGNITUD_PATH = "data/gold/dim_magnitud.parquet"
DIM_MEDIDA_TRAFICO_PATH = "data/gold/dim_medida_trafico.parquet"
DIM_PUNTO_TRAFICO_PATH = "data/gold/dim_punto_trafico.parquet"
DIM_FECHA_PATH = "data/gold/dim_fecha.parquet"
FACT_CALIDAD_AIRE_PATH = "data/gold/fact_calidad_aire.parquet"
FACT_TRAFICO_DIARIO_PATH = "data/gold/fact_trafico_diario.parquet"

## Dimensions

Small -- loaded entirely with pandas.

In [2]:
dim_distrito = pd.read_parquet(DIM_DISTRITO_PATH)
dim_distrito.shape, dim_distrito.dtypes

((22, 11),
 COD_DIS               object
 NOMBRE                object
 DISTRI_MAY            object
 DISTRI_MT             object
 AREA                 float64
 GEOMETRY              object
 GEOMETRY_WKT          object
 N_ESTACIONES_AIRE      int64
 N_PUNTOS_TRAFICO       int64
 COBERTURA_AIRE          bool
 COBERTURA_TRAFICO       bool
 dtype: object)

In [3]:
dim_distrito.head(5)

,COD_DIS,NOMBRE,DISTRI_MAY,DISTRI_MT,AREA,GEOMETRY,GEOMETRY_WKT,N_ESTACIONES_AIRE,N_PUNTOS_TRAFICO,COBERTURA_AIRE,COBERTURA_TRAFICO
0,01,Centro,CENTRO,CENTRO,5.228246e+06,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00q\x00\x0...,"POLYGON ((441184.7842 4473200.7799, 441181.244...",1,166,True,True
1,02,Arganzuela,ARGANZUELA,ARGANZUELA,6.462176e+06,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x9f\x00...,"POLYGON ((440272.9189 4472969.5669, 440350.318...",1,259,True,True
2,03,Retiro,RETIRO,RETIRO,5.466211e+06,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00g\x00\x0...,"POLYGON ((443652.1493 4473312.5997, 443560.739...",1,171,True,True
3,04,Salamanca,SALAMANCA,SALAMANCA,5.392404e+06,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00a\x00\x0...,"POLYGON ((444075.1781 4476611.8274, 444057.807...",1,217,True,True
4,05,Chamartín,CHAMARTIN,CHAMARTÍN,9.175482e+06,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x96\x00...,"POLYGON ((442904.1141 4481525.2122, 442901.019...",3,362,True,True


In [4]:
dim_estacion_aire = pd.read_parquet(DIM_ESTACION_AIRE_PATH)
dim_estacion_aire.shape, dim_estacion_aire.dtypes

((24, 13),
 ID_AIRE        int64
 ESTACION      object
 DIRECCION     object
 LATITUD      float64
 LONGITUD     float64
 COD_DIS       object
 NO2            int64
 SO2            int64
 CO             int64
 PM10           int64
 PM2_5          int64
 O3             int64
 BTX            int64
 dtype: object)

In [5]:
dim_estacion_aire.head(5)

,ID_AIRE,ESTACION,DIRECCION,LATITUD,LONGITUD,COD_DIS,NO2,SO2,CO,PM10,PM2_5,O3,BTX
0,4,Plaza de España,Plaza de España,40.423882,-3.712257,09,1,1,1,0,0,0,0
1,8,Escuelas Aguirre,Entre C/ Alcalá y C/ O’ Donell,40.421553,-3.682316,04,1,1,1,1,1,1,1
2,11,Ramón y Cajal,Avda. Ramón y Cajal esq. C/ Príncipe de Vergara,40.451473,-3.677349,05,1,0,0,0,0,0,1
3,16,Arturo Soria,C/ Arturo Soria esq. C/ Vizconde de los Asilos,40.440046,-3.639242,15,1,0,0,0,0,1,0
4,17,Villaverde,C/ Juan Peñalver,40.347147,-3.713317,17,1,0,0,0,0,1,0


In [6]:
dim_magnitud = pd.read_parquet(DIM_MAGNITUD_PATH)
dim_magnitud.shape, dim_magnitud.dtypes

((18, 2),
 ID_MAGNITUD     int32
 MAGNITUD       object
 dtype: object)

In [7]:
dim_magnitud.head(5)

,ID_MAGNITUD,MAGNITUD
0,1,SO2
1,6,CO
2,7,NO
3,8,NO2
4,9,PM2.5


In [8]:
dim_punto_trafico = pd.read_parquet(DIM_PUNTO_TRAFICO_PATH)
dim_punto_trafico.shape, dim_punto_trafico.dtypes

((5081, 5),
 ID_TRAFICO      int64
 COD_DIS        object
 NOMBRE         object
 LATITUD       float64
 LONGITUD      float64
 dtype: object)

In [9]:
dim_punto_trafico.head(5)

,ID_TRAFICO,COD_DIS,NOMBRE,LATITUD,LONGITUD
0,1001,10,05FT10PM01,40.409729,-3.740786
1,1002,10,05FT37PM01,40.408030,-3.743760
2,1003,10,05FT66PM01,40.406824,-3.746834
3,1006,10,04FT74PM01,40.411894,-3.736324
4,1009,09,03FT52PM01,40.416234,-3.724909


## `dim_medida_trafico`

New catalog (this session) -- lets `fact_trafico_diario` unpivot INTENSIDAD/OCUPACION/CARGA/VMED into rows, mirroring how `dim_magnitud` unpivots gases in `fact_calidad_aire`.

In [10]:
dim_medida_trafico = pd.read_parquet(DIM_MEDIDA_TRAFICO_PATH)
dim_medida_trafico

,ID_MEDIDA_TRAFICO,MEDIDA,UNIDAD
0,1,INTENSIDAD,veh/h
1,2,OCUPACION,%
2,3,CARGA,0-100
3,4,VMED,km/h


## `dim_fecha`

One row per calendar day, spans the date range of both facts. Built from silver, not from the facts, to avoid a build-order dependency.

In [11]:
dim_fecha = pd.read_parquet(DIM_FECHA_PATH)
dim_fecha.shape, dim_fecha.dtypes

((1277, 6),
 FECHA            datetime64[us]
 ANIO                      int64
 MES                       int64
 NOMBRE_MES               object
 DIA_SEMANA               object
 ES_FIN_SEMANA              bool
 dtype: object)

In [12]:
dim_fecha.head(5)

,FECHA,ANIO,MES,NOMBRE_MES,DIA_SEMANA,ES_FIN_SEMANA
0,2023-01-01,2023,1,January,Sunday,True
1,2023-01-02,2023,1,January,Monday,False
2,2023-01-03,2023,1,January,Tuesday,False
3,2023-01-04,2023,1,January,Wednesday,False
4,2023-01-05,2023,1,January,Thursday,False


## Fact

> Rango de fechas de cada fact. DuckDB lee solo la columna `FECHA` del parquet
> (projection pushdown), así que no hace falta cargar las 100M+ filas en memoria.
>
> `dias_con_dato` vs `dias_del_rango` es lo interesante: si no coinciden, hay
> huecos dentro del rango (no es un histórico continuo).

In [5]:
duckdb.sql(f"""
    SELECT
        min(FECHA)                                  AS fecha_min,
        max(FECHA)                                  AS fecha_max,
        count(DISTINCT FECHA)                       AS dias_con_dato,
        datediff('day', min(FECHA), max(FECHA)) + 1 AS dias_del_rango,
        count(*)                                    AS filas
    FROM '{FACT_CALIDAD_AIRE_PATH}'
""").df()

,fecha_min,fecha_max,dias_con_dato,dias_del_rango,filas
0,2023-01-01,2026-06-30,1245,1277,157741


In [4]:
duckdb.sql(f"""
    SELECT
        min(FECHA)                                  AS fecha_min,
        max(FECHA)                                  AS fecha_max,
        count(DISTINCT FECHA)                       AS dias_con_dato,
        datediff('day', min(FECHA), max(FECHA)) + 1 AS dias_del_rango,
        count(*)                                    AS filas
    FROM '{FACT_TRAFICO_DIARIO_PATH}'
""").df()

,fecha_min,fecha_max,dias_con_dato,dias_del_rango,filas
0,2023-01-01,2025-12-31,212,1096,3834380
